In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from src.utils import * 
import src.prompt as prompt
from src.data_loader import load_spatial_data_anndata

import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# data

In [ ]:
representetive_gene_list = ["Aqp4", "Hpcal1", "Pvalb", "Frem3", "Pcp4", "Krt17","Mobp", 
                          "Lamp5", "Rorb", "Fezf2", "Syt6", "Fa2h",
                          "Plp1", "Foxj1", "Gfap", "Cpne5", "Kcnip2",
                          "Bgn", "Cux2", "Etv1",
                          "Zmat4", "Rab3c"]
representetive_gene_list = [gene.upper() for gene in representetive_gene_list]

In [ ]:
config = load_config("configs/config_finetunePro_libd.yaml")
config.data_name = "151673"
config.refresh_paths()

domain_mapping = {1: "Layer1", 2: "Layer2", 3: "Layer3", 4: "Layer4", 5: "Layer5", 6: "Layer6", 7: "WM"}


In [ ]:
# --- Load data ---
data_path = str(dataset_dir("visium_libd", config.data_name))

adata = load_spatial_data_anndata(
    data_path=data_path,
    adata_file="filtered_feature_bc_matrix.h5",
    config=config,
    celltype_path=os.path.join("examples/visium_libd/deconv_result", f"celltype_proportions_{config.data_name}.csv"),
    optional_files=["metadata.tsv"]
)
# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)


## sample data

In [ ]:
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = config.prototype_p

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


train_neighbor_normalized_df_genes, val_neighbor_normalized_df_genes = train_test_split(neighbor_normalized_df_genes, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )

# prepare ground truth for calculate NMI and ARI and other metrics
# this need to be done for each dataset
# 151673

truth_df = adata.obs.loc[val_neighbor_normalized_df.index]
truth_df.to_csv("examples/intermediates/local_llm_results/LIBD/151673_test_all_ground_truth.csv")

In [ ]:
# check sample distribution
adata.obs[config.name_truth].loc[train_neighbor_normalized_df.index].value_counts()

In [ ]:
# prototype
# calculate prototype
train_neighbor_df = train_neighbor_normalized_df.join(train_neighbor_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[config.name_truth].loc[train_neighbor_df.index], train_neighbor_df], axis=1).groupby(config.name_truth, observed=False).mean()
print(one_shot_df.index)


In [ ]:
# validation in finetune is not necessary
# # finetune train and val data
# _, val_for_finetune = train_test_split(val_neighbor_normalized_df, 
#                                                             test_size=0.1, 
#                                                             random_state=seed
#                                                            )

# adata.obs.loc[val_for_finetune.index, config.name_truth].value_counts()

# prompt

In [ ]:
# unique_layers = adata.obs[name_truth].unique()
# domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping


config.cell_names = neighbor_normalized_df.columns
config.gene_names = neighbor_normalized_df_genes.columns


# generate Comparison-based Prompt
config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)


In [ ]:
print(config.system_prompt)


In [ ]:
print(prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, 1, config))
print(prompt.finetune_assistant(train_neighbor_normalized_df, 1, adata.obs[config.name_truth]))


# GPT-4o-mini

In [ ]:
print(f"finetune_json/{config.data_name}_{config.model_type}/")
print(config.replicate)

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, i, config)
        assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

# val_output_file = f"{output_folder}{config.data_name}_val_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
# print(f"Generating json for validation into {val_output_file}")
# with open(val_output_file, 'w') as f:
#     for i in range(val_for_finetune.shape[0]):
#         system_p = config.system_prompt
#         user_p = prompt.finetune_user_deconv(val_for_finetune, i, config)
#         assistant_p = prompt.finetune_assistant(val_for_finetune, i, adata.obs[config.name_truth])

#         row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
#         json_str = json.dumps(row_data)
#         f.write(json_str + '\n')  

In [ ]:
print(system_p + user_p + "\n" + assistant_p)


In [ ]:
train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)

# val_file = client.files.create(
#   file=open(val_output_file, "rb"),
#   purpose="fine-tune"
# )

finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  # validation_file=val_file.id,
  model="gpt-4o-mini-2024-07-18",
  suffix=f"{config.model_type}_{config.r}"  # default is empty
)


# Local LLM

In [ ]:
# prepare dataset for batch inference
# this test_output_file is used for batch inference
# using the whole val_neighbor_normalized_df to test the model rather than 10% 
test_output_file = f"{output_folder}{config.data_name}_test_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for test into {test_output_file}")
with open(test_output_file, 'w') as f:
    for i in range(val_neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype_geneorder(val_neighbor_normalized_df, val_neighbor_normalized_df_genes, i, config)
        assistant_p = ""
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

In [ ]:
# 151674, 151675, 151676
config.data_name = "151676"
config.replicate = "_localllm" 
config.refresh_paths()

# --- Load data ---
data_path = str(dataset_dir("visium_libd", config.data_name))

adata = load_spatial_data_anndata(
    data_path=data_path,
    adata_file="filtered_feature_bc_matrix.h5",
    config=config,
    celltype_path=os.path.join("examples/visium_libd/deconv_result", f"celltype_proportions_{config.data_name}.csv"),
    optional_files=["metadata.tsv"]
)
# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)
# prepare ground truth for calculate NMI and ARI and other metrics
# this need to be done for each dataset
truth_df = adata.obs
truth_df.to_csv(f"examples/intermediates/local_llm_results/LIBD/{config.data_name}_all_ground_truth.csv")


cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping


config.cell_names = neighbor_normalized_df.columns
config.gene_names = neighbor_normalized_df_genes.columns

# generate json for finetune
output_folder = f"batch_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

output_file = f"{output_folder}{config.data_name}_all_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {output_file}")
with open(output_file, 'w') as f:
    for i in range(neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype_geneorder(neighbor_normalized_df, neighbor_normalized_df_genes, i, config)
        assistant_p = ""
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')

## refine results

In [ ]:
local_LLM_results_151673 = pd.read_csv("examples/intermediates/local_llm_results/processed/tmp/151673_test_all_results.csv")
ground_truth_151673 = pd.read_csv("examples/intermediates/local_llm_results/LIBD/151673_test_all_ground_truth.csv",index_col=0)
local_LLM_results_151674 = pd.read_csv("examples/intermediates/local_llm_results/processed/tmp/151674_all_all_results.csv")
ground_truth_151674 = pd.read_csv("examples/intermediates/local_llm_results/LIBD/151674_all_ground_truth.csv", index_col=0)
local_LLM_results_151675 = pd.read_csv("examples/intermediates/local_llm_results/processed/tmp/151675_all_all_results.csv")
ground_truth_151675 = pd.read_csv("examples/intermediates/local_llm_results/LIBD/151675_all_ground_truth.csv", index_col=0)
local_LLM_results_151676 = pd.read_csv("examples/intermediates/local_llm_results/processed/tmp/151676_all_all_results.csv")
ground_truth_151676 = pd.read_csv("examples/intermediates/local_llm_results/LIBD/151676_all_ground_truth.csv", index_col=0)


In [ ]:
local_LLM_all_in_one_reaults = pd.DataFrame()
data_types = ["libd"]
data_names = ["151673_test", "151674_all", "151675_all", "151676_all"]  # , "BZ9_all", "BZ14_all"
model_names = ["Llama8", "Qwen30", "Llama70"]  # , "Qwen30", "Llama70"
location_key = config.pos_name
truth_key = config.name_truth
reps = ["rep1", "rep2", "rep3"]  # , "rep2", "rep3"
experiment_types = ["zeroshot", "finetune_BZ5"]

for data_type in data_types:
    for data in data_names:
        for model in model_names:
            for experiment_type in experiment_types:
                for rep in reps:
                    if data == "151673_test":
                        obs_df = ground_truth_151673.copy()
                        local_llm_results_df = local_LLM_results_151673.copy()
                    elif data == "151674_all":
                        obs_df = ground_truth_151674.copy()
                        local_llm_results_df = local_LLM_results_151674.copy()
                    elif data == "151675_all":
                        obs_df = ground_truth_151675.copy()
                        local_llm_results_df = local_LLM_results_151675.copy()
                    elif data == "151676_all":
                        obs_df = ground_truth_151676.copy()
                        local_llm_results_df = local_LLM_results_151676.copy()
                    setting_results = local_llm_results_df[
                        (local_llm_results_df['model'] == model) & 
                        (local_llm_results_df['experiment_type'] == experiment_type) & 
                        (local_llm_results_df['data_name'] == data) &
                        (local_llm_results_df['replicate'] == rep)
                    ].copy()
                    if len(setting_results) == 0:
                        continue
                    #print(obs_df.columns)
                    setting_results.index = obs_df.index

                    # drop useless columns in obs_df
                    obs_df = obs_df.drop(columns=['replicate'])


                    obs_df = obs_df.join(setting_results, how='left')

                    # refine results
                    adj_matrix, _ = sparse_adjacency(obs_df[location_key], threshold=config.r)
                    refined_niche = relabel_cells(adj_matrix.toarray(), obs_df['prediction'])
                    obs_df['local_llm_result_refined'] = refined_niche

                    local_LLM_all_in_one_reaults = pd.concat([local_LLM_all_in_one_reaults, obs_df])




In [ ]:
local_LLM_all_in_one_reaults.to_csv("examples/results/localllm_libd_all_in_one_results.csv")


# gemini 1.5 flash

In [ ]:
import google

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["API_KEY"])
for model_info in genai.list_tuned_models():
    print(model_info.name)

In [ ]:

tunable_models = [
    m for m in genai.list_models()
    if "createTunedModel" in m.supported_generation_methods]
tunable_models

In [ ]:
base_model = "models/gemini-1.5-flash-001-tuning"
training_data = []
for i in range(train_neighbor_normalized_df.shape[0]):
    system_p = prompt.finetune_system_deconv(config)
    user_p = prompt.finetune_user_deconv(train_neighbor_normalized_df, i, config)
    assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
    training_data.append({'text_input': system_p + user_p, 'output': assistant_p})
    



In [ ]:
training_data[:4]

In [ ]:
operation = genai.create_tuned_model(
    # You can use a tuned model here too. Set `source_model="tunedModels/..."`
    display_name=config.data_name,
    source_model=base_model,
    epoch_count=30,
    batch_size=4,
    learning_rate=0.001,
    training_data=training_data,
)

In [ ]:
for status in operation.wait_bar():
    time.sleep(10)

In [ ]:
result = operation.result()

In [ ]:
import seaborn as sns
model = operation.result()  # model = genai.get_tuned_model()
snapshots = pd.DataFrame(model.tuning_task.snapshots)

sns.lineplot(data=snapshots, x = 'epoch', y='mean_loss')

# Use the finetuned model

## generate json

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetunePro_libd.yaml")
config.data_name = "151507"
config.refresh_paths()
name_truth = config.name_truth

unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping


config.cell_names = neighbor_normalized_df.columns
config.gene_names = common_genes


# generate Comparison-based Prompt
config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)




In [ ]:
print(config.folder_path)

In [ ]:
config.replicate = '_rep3R600'


In [ ]:
print(config.gpt_model)

In [ ]:
# make sure the gpt_model is correct
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=3000, df_extra=val_neighbor_normalized_df_genes)

## submit

In [ ]:

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_libd.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
# Replace strings in the first column of gpt_results_df that contain keywords plus '**' or '.'
import re

# Get the list of keywords from domain_mapping
keywords = list(domain_mapping.values())

# Create a regex pattern to capture any keyword possibly surrounded by other text
pattern = r'.*(' + '|'.join(map(re.escape, keywords)) + r')[\*\.\s]*.*'

# Replace the entire string with the captured keyword only if it could be not unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<5].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"].str.replace(pattern, r'\1', regex=True)


In [ ]:
gpt_results_df.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "mpa", "finetunePro_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Mpa", "finetunePro_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "FX", "finetunePro_gpt4o_mini"] = "fx"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Outputs: pvh", "finetunePro_gpt4o_mini"] = "PVH"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "pv", "finetunePro_gpt4o_mini"] = "PV"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Pv", "finetunePro_gpt4o_mini"] = "PV"

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<5].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
# # manually retrieve batch output
# output_file_name = f"{output_path}/response_BZ5_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}_{with_count_numbers}.txt"
# batch_id = "batch_66f62020d11c81909424c1423da11a70"
# file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# # Open the file in write mode and save the string
# with open(output_file_name, 'w') as file:
#     file.write(file_response.text)

## plot and save

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()
val_adata.obs = val_adata.obs.join(gpt_results_df)
val_adata.obs['finetunePro_gpt4o_mini'] = val_adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")
sc.pl.spatial(val_adata, color="finetunePro_gpt4o_mini", title =  f"finetunePro_gpt4o_mini")

print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini']))
print(normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini']))

In [ ]:
# print(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



## refine the niche

In [ ]:
config.r_factor = 1
val_adj_matrix, _ = sparse_adjacency(pos_data.loc[val_neighbor_normalized_df.index], threshold=r*config.r_factor)
refined_niche = relabel_cells(val_adj_matrix.toarray(), gpt_results_df['finetunePro_gpt4o_mini'])

In [ ]:
val_adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
sc.pl.spatial(val_adata, color="finetunePro_gpt4o_mini_refined", title =  f"finetunePro_gpt4o_mini_refined")
print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined']))
print(normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined']))

In [ ]:
val_adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


# test

## load test data 
Prototype in prompt is based on training data

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetunePro_libd.yaml")
config.data_name = "151671"
config.refresh_paths()
name_truth = config.name_truth



### this is codes for loading DLPFC data

In [ ]:
# # 
# name_truth = "ManualAnnotation"
# config.name_truth = name_truth
# deconv_result_dir = "examples/visium_libd/deconv_result/"



# # --- Load data ---
# print(f"processing data: {config.data_name}")
# # --- Load data ---
# dataset_folder = "data/processed_inputs/spatialDLPFC_new"
# data_path = f"{dataset_folder}/adata_vis_orig.h5ad"
# adata = sc.read_h5ad(data_path)
# adata =  adata[adata.obs['sample_id'] == config.data_name].copy()

# adata = adata[:, adata.var['gene_type'] == "protein_coding"].copy()
# adata.var_names = adata.var['SYMBOL'].astype(str)
# adata.var_names_make_unique()

# sc.pp.filter_genes(adata, min_cells=10)
# sc.pp.normalize_total(adata, inplace=True)
# sc.pp.log1p(adata)
# sc.pp.scale(adata)

# # remove the name_truth in adata.obs because they are NA
# adata.obs = adata.obs.drop(columns=[name_truth])

# domain_data = pd.read_csv(f"{dataset_folder}/05-shared_utilities/nonIF/spatialLIBD_ManualAnnotation_{config.data_name}_layers.csv", index_col=None)
# domain_data.index = domain_data['spot_name'] + '_' + domain_data['sample_id']
# domain_data = domain_data.drop(columns=['spot_name', 'sample_id'])

# adata.obs = adata.obs.join(domain_data)
# # remove the obs that are na in name_truth
# adata = adata[~adata.obs[name_truth].isna()].copy()

# cell_proportion_data = pd.read_csv(os.path.join(deconv_result_dir, f"celltype_proportions_{config.data_name}.csv"), index_col=0)
# adata.obs = adata.obs.join(cell_proportion_data)

# celltype_names = cell_proportion_data.columns

# # clean the cell ID to save token
# n_cells = len(adata)
# adata.obs_names = [str(i) for i in range(n_cells)]
# pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)

# cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()

# common_genes = list(set(adata.var_names) & set(important_marker_genes))
# adata = adata[:, common_genes]

### this is codes for loading LIBD data

In [ ]:
# this is codes for loading LIBD data
# --- Load data ---
print(f"processing data: {config.data_name}")
data_path = str(dataset_dir("visium_libd", config.data_name))
# marker_path = "data/reference/marker_genes/Human_cell_markers.txt"
adata = sc.read_visium(data_path)
adata.var_names_make_unique()

cell_proportion_data = pd.read_csv(os.path.join("examples/visium_libd/deconv_result", f"celltype_proportions_{config.data_name}.csv"), index_col=0)

adata.obs = adata.obs.join(cell_proportion_data)

# Normalize data
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)



common_genes = list(set(adata.var_names) & set(important_marker_genes))
adata = adata[:, common_genes]

# read the metadata
meta_data = pd.read_csv(os.path.join(data_path, "metadata.tsv"), sep="\t")
# merge the metadata to adata
adata.obs = adata.obs.merge(meta_data, left_index=True, right_index=True, how="left")
# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

# clean the cell ID to save token
# Rename the obs_names of adata
adata.obs_names = [f'spot_{i}' for i in range(len(adata.obs_names))]

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
cell_proportion_data = adata.obs[cell_proportion_data.columns].copy()





In [ ]:
# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r, add_diagonal=True)

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(cell_proportion_data)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
print(f"Mean of n_neighbors: {np.mean(n_neighbors)}")
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized, 
                              index=adata.obs_names,
                              columns=cell_proportion_data.columns)

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata.X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=adata.var_names)


## test GPT

In [ ]:
config.replicate="_testK7"


In [ ]:

# unique_layers = adata.obs[name_truth].unique()
# domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

cell_names_mapping ={'Astro': 'Astrocyte',
 'EndoMural': 'Endothelial and mural cells',
 'Excit_L2_3': 'Excitatory neuron layer 2/3',
 'Excit_L3': 'Excitatory neuron layer 3',
 'Excit_L3_4_5': 'Excitatory neuron layer 3/4/5',
 'Excit_L4': 'Excitatory neuron layer 4',
 'Excit_L5': 'Excitatory neuron layer 5',
 'Excit_L5_6': 'Excitatory neuron layer 5/6',
 'Excit_L6': 'Excitatory neuron layer 6',
 'Inhib': 'Inhibitory neuron',
 'Micro': 'Microglia',
 'OPC': 'Oligodendrocyte precursor cell',
 'Oligo': 'Oligodendrocyte'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping


config.cell_names = neighbor_normalized_df.columns
config.gene_names = common_genes


# generate Comparison-based Prompt
config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)


In [ ]:
print(config.data_name)
print(config.replicate)
print(config.gpt_model)

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=3000, df_extra=neighbor_normalized_df_genes)

In [ ]:
# submit_end2end.py

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_libd.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ").replace("‘", "'").replace("’", "'")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
gpt_results_df

In [ ]:
gpt_results_df = gpt_results_df[[0]]
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
# gpt_results_df = pd.read_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
# gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "LayerWM", "finetunePro_gpt4o_mini"] = "WM"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Inputs: 'Layer5'", "finetunePro_gpt4o_mini"] = "Layer5"


In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<5].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

## plot and save

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['finetune_gpt4o'] with "unknown"
adata.obs['finetunePro_gpt4o_mini'] = adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")
sc.pl.spatial(adata, color="finetunePro_gpt4o_mini",library_id=config.data_name, title =  f"finetunePro_gpt4o_mini")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))

In [ ]:
# refine the niche
config.r_factor = 1
refined_niche = relabel_cells(adj_matrix.toarray(), gpt_results_df['finetunePro_gpt4o_mini'])
adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_refined", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_refined")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))

In [ ]:
save_folder = "./finetune_results/LIBD"
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
print(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# Run this in second stage
save_folder = "./finetune_results/LIBD"
adata.obs = pd.read_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
adata.obs.index = adata.obs.index.astype(str)

## high confidence cell

### get confident cells

In [ ]:
label_key = 'finetunePro_gpt4o_mini'  # IMPORTANT!!!!! before refinement
high_confidence_mask = find_high_confidence_cells(
    adata,
    label_key=label_key,
    k=min(30, int((np.mean(n_neighbors))/2)),  # Consider top 20/ half of the mean nearest neighbors  # IMPORTANT!!!!!!  others are min(20) only merfish29 is min(30)
    distance_threshold=config.r
)

In [ ]:
# adata.uns.pop('confident_cells_colors')

In [ ]:
adata.obs['confident_cells'] = adata.obs['finetunePro_gpt4o_mini'].copy()
adata.obs['confident_cells'] = adata.obs['confident_cells'].astype(str)
adata.obs.loc[~high_confidence_mask, 'confident_cells'] = 'unconfident'
sc.pl.spatial(adata, color="confident_cells", library_id=config.data_name, title =  f"confident_cells")


In [ ]:
# Get unique values excluding 'unconfident'
confident_unique = set(adata.obs.loc[high_confidence_mask, 'confident_cells'].unique()) - {'unconfident'}
unique_layers = set(adata.obs[label_key].unique()) - {'unknown'}
# Check if any elements are missing
missing_elements = unique_layers - confident_unique
old_one_shot_df = pd.DataFrame()
if len(missing_elements) > 0:
    print(f"Missing elements in confident cells: {missing_elements}")
    print("use the old one_shot_df for that niche")
    old_one_shot_df = one_shot_df.loc[list(missing_elements)]
else:
    print("No missing elements in confident cells") 

In [ ]:
old_one_shot_df

### prototype for second stage

In [ ]:


conserved_normalized_df = neighbor_normalized_df.loc[high_confidence_mask]
conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[high_confidence_mask]

un_conserved_normalized_df = neighbor_normalized_df.loc[~high_confidence_mask]
un_conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[~high_confidence_mask]   

print(f"find {len(conserved_normalized_df)} conserved cells")
print(f"find {len(un_conserved_normalized_df)} un-conserved cells")

In [ ]:
print(f"conserved_json/{config.data_name}_{config.model_type}/" )

In [ ]:
conserved_neighbor_df = conserved_normalized_df.join(conserved_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[label_key].loc[conserved_neighbor_df.index], conserved_neighbor_df], axis=1).groupby(label_key, observed=False).mean()
# remove unknown
if 'unknown' in one_shot_df.index:
    one_shot_df = one_shot_df.drop(index='unknown')

In [ ]:
# remove nan in one_shot_df
one_shot_df = one_shot_df.dropna()
one_shot_df

In [ ]:
# IMPORTANT!!!!!
# prototype is from conserved cells

# calculate prototype
conserved_neighbor_df = conserved_normalized_df.join(conserved_normalized_df_genes).copy()
one_shot_df = pd.concat([adata.obs[label_key].loc[conserved_neighbor_df.index], conserved_neighbor_df], axis=1).groupby(label_key, observed=False).mean()
one_shot_df = one_shot_df.dropna()

# remove unknown
if 'unknown' in one_shot_df.index:
    one_shot_df = one_shot_df.drop(index='unknown')
if len(old_one_shot_df) > 0:
    one_shot_df = pd.concat([old_one_shot_df, one_shot_df], axis=0)

config.system_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes

In [ ]:
print(config.system_prompt)

### oneshot with conserved cells

In [ ]:
print(config.folder_path)
print(config.gpt_model)
config.replicate = "_rep2R600_unconserved"  # "_rep2R100_unconserved" is for one shot, _rep2R100_finetune is for finetune
print(config.replicate)

In [ ]:
# IMPORTANT!!!!!
# json is from un-conserved cells
generate_json_end2end(un_conserved_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, max_completion_tokens=128, batch_size=3000, df_extra=un_conserved_normalized_df_genes)


In [ ]:
# submit_end2end.py

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_libd.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro_73{config.replicate}_{config.prototype_p}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
if len(un_conserved_normalized_df) > 3000:
    n_batch = 2
else:
    n_batch = 1

for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.finetunePro_gpt4o_mini.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "LayerWM", "finetunePro_gpt4o_mini"] = "WM"
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Layer.4", "finetunePro_gpt4o_mini"] = "Layer4"

In [ ]:
# fill nan with unknown
gpt_results_df.fillna("unknown", inplace=True)

In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<5].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
# update the finetunePro_gpt4o_mini with second stage results
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs[label_key].copy()
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs['finetunePro_gpt4o_mini_twostage'].astype(str)
adata.obs.loc[gpt_results_df.index, 'finetunePro_gpt4o_mini_twostage'] = gpt_results_df.finetunePro_gpt4o_mini

In [ ]:
sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_twostage", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_twostage")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage']))

In [ ]:
# refine the niche
adj_matrix, _ = sparse_adjacency(pos_data.loc[neighbor_normalized_df.index], threshold=r)

refined_niche = relabel_cells(adj_matrix.toarray(), adata.obs['finetunePro_gpt4o_mini_twostage'])
adata.obs['finetunePro_gpt4o_mini_twostage_refined'] = refined_niche


In [ ]:
sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_twostage_refined", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_twostage_refined")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage_refined']))
save_folder = './twostage_results/LIBD'
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
config.data_name

# plot for paper

In [ ]:

adata.obs = pd.read_csv(f"./twostage_results/LIBD/folder_1516730.3/151671_with_refined_finetunePro_1516730.3_False_False_True_countPlusGenes_False_False_False_True_rep2R600_unconserved.csv")


In [ ]:
sc.pl.spatial(adata, 
              color=["finetunePro_gpt4o_mini_refined", "finetunePro_gpt4o_mini_twostage_refined", config.name_truth], 
              library_id=config.data_name, 
              title =  ["Spec1First", "Spec1Second", "Ground Truth"],
              save="151671_Spec1First_Spec1Second_GroundTruth.pdf"
              )

In [ ]:
adata.uns['finetunePro_gpt4o_mini_twostage_refined_colors'] = [
 '#2ca02c',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2']

adata.uns['finetunePro_gpt4o_mini_refined_colors'] = [
 '#2ca02c',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2']

adata.uns['layer_guess_colors'] = [
 '#2ca02c',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2']

In [ ]:
adata.uns.keys()

In [ ]:
['#1f77b4',
 '#ff7f0e',
 '#2ca02c',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2',
 '#7f7f7f']